In [ ]:
!pip install unsloth

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.2/81.2 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.1/23.1 MB 89.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 22.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 44.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 110.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 36.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 84.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 106.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 101.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.9/216.9 kB 22.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.7/119.7 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 199

In [ ]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
Tesla T4


In [13]:
from google.colab import userdata
import openai
client = openai.OpenAI(api_key=userdata.get('OPENAI_API_KEY'))

In [14]:
import json

def generate_qa_pairs(source_text, n_pairs=20):
    prompt = f"""You are creating training data for fine-tuning an AI model on FDA/GxP regulatory compliance knowledge.

Read the source text below and generate exactly {n_pairs} question-answer pairs based ONLY on facts in this text. Do not invent information not present in the source.

Rules:
- Questions should be realistic things a compliance professional would ask
- Answers should be accurate, concise (2-4 sentences), and professional in tone
- Vary question types: definitions, "what does X require", "how do I comply with Y", scenario-based

Return ONLY a JSON array, no other text, in this exact format:
[{{"instruction": "question here", "input": "", "output": "answer here"}}]

Source text:
{source_text}
"""
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.7
    )
    raw = response.choices[0].message.content
    # strip markdown code fences if the model adds them
    raw = raw.strip().removeprefix("```json").removeprefix("```").removesuffix("```").strip()
    return json.loads(raw)

In [15]:
qa_pairs = generate_qa_pairs(source_text, n_pairs=20)
print(len(qa_pairs))
print(qa_pairs[0])  # inspect one example

19
{'instruction': 'What do closed systems require for electronic records?', 'input': '', 'output': 'Closed systems must employ procedures and controls to ensure the authenticity, integrity, and confidentiality of electronic records, ensuring that signers cannot easily repudiate the signed records.'}


In [8]:
source_text = """
Persons who use closed systems to create, modify, maintain, or transmit electronic records shall employ procedures and controls designed to ensure the authenticity, integrity, and, when appropriate, the confidentiality of electronic records, and to ensure that the signer cannot readily repudiate the signed record as not genuine. Such procedures and controls shall include the following:
(a) Validation of systems to ensure accuracy, reliability, consistent intended performance, and the ability to discern invalid or altered records.
(b) The ability to generate accurate and complete copies of records in both human readable and electronic form suitable for inspection, review, and copying by the agency.
(c) Protection of records to enable their accurate and ready retrieval throughout the records retention period.
"""



In [16]:
source_chunks = [
"""
Persons who use closed systems to create, modify, maintain, or transmit electronic records shall employ procedures and controls designed to ensure the authenticity, integrity, and, when appropriate, the confidentiality of electronic records, and to ensure that the signer cannot readily repudiate the signed record as not genuine. Such procedures and controls shall include the following:
(a) Validation of systems to ensure accuracy, reliability, consistent intended performance, and the ability to discern invalid or altered records.
(b) The ability to generate accurate and complete copies of records in both human readable and electronic form suitable for inspection, review, and copying by the agency.
(c) Protection of records to enable their accurate and ready retrieval throughout the records retention period.
"""
    """(d) Limiting system access to authorized individuals.""",
    """(e) Use of secure, computer-generated, time-stamped audit trails to independently record the date and time of operator entries and actions that create, modify, or delete electronic records. Record changes shall not obscure previously recorded information. Such audit trail documentation shall be retained for a period at least as long as that required for the subject electronic records and shall be available for agency review and copying.""",
    """Use of operational system checks to enforce permitted sequencing of steps and events, as appropriate.""",
    """Use of authority checks to ensure that only authorized individuals can use the system, electronically sign a record, access the operation or computer system input or output device, alter a record, or perform the operation at hand.""",
]

In [17]:
import time

all_qa_pairs = []

for i, chunk in enumerate(source_chunks):
    print(f"Generating from chunk {i+1}/{len(source_chunks)}...")
    pairs = generate_qa_pairs(chunk, n_pairs=25)
    all_qa_pairs.extend(pairs)
    time.sleep(2)  # small pause, avoids hitting rate limits

print(f"Total pairs generated: {len(all_qa_pairs)}")

Generating from chunk 1/4...
Generating from chunk 2/4...
Generating from chunk 3/4...
Generating from chunk 4/4...
Total pairs generated: 91


In [18]:
import random
for pair in random.sample(all_qa_pairs, 5):
    print("Q:", pair['instruction'])
    print("A:", pair['output'])
    print("---")

Q: How can I confirm the reliability of my electronic record systems?
A: To confirm the reliability of electronic record systems, conduct regular validations, perform system checks, and ensure that the systems consistently perform their intended functions without errors.
---
Q: What is the significance of generating copies in both forms (human-readable and electronic)?
A: Generating copies in both human-readable and electronic forms ensures that records can be accessed and reviewed by various stakeholders, including regulatory agencies, facilitating compliance and audit processes.
---
Q: What action must be taken when an electronic record is deleted?
A: When an electronic record is deleted, the audit trail must still reflect the action and document the date and time of the deletion.
---
Q: How can I ensure compliance with authority checks?
A: To ensure compliance with authority checks, establish clear policies for user access, regularly audit user permissions, and provide training on t

In [19]:
with open('qa_pairs.jsonl', 'w') as f:
    for pair in all_qa_pairs:
        f.write(json.dumps(pair) + '\n')

print("Saved", len(all_qa_pairs), "pairs")

Saved 91 pairs


In [20]:
print(f"Total pairs generated: {len(all_qa_pairs)}")

Total pairs generated: 91


In [21]:
import random
for pair in random.sample(all_qa_pairs, 5):
    print("Q:", pair['instruction'])
    print("A:", pair['output'])
    print("---")

Q: How should organizations manage access to audit trails?
A: Organizations should implement strict access controls to manage who can view or modify audit trails, ensuring that only authorized personnel can access this sensitive information.
---
Q: What is the purpose of time-stamped audit trails?
A: Time-stamped audit trails are used to independently record operator actions, providing a clear timeline of when specific entries and modifications to electronic records were made.
---
Q: What type of systems require operational system checks?
A: Operational system checks are typically required in systems where the sequence of operations is critical to compliance, such as in manufacturing, laboratory procedures, and quality control processes.
---
Q: What role does documentation play in FDA/GxP compliance?
A: Documentation plays a critical role in FDA/GxP compliance as it provides evidence of adherence to regulations and helps ensure the integrity and reliability of electronic records.
---
Q

In [22]:
with open('qa_pairs.jsonl', 'w') as f:
    for pair in all_qa_pairs:
        f.write(json.dumps(pair) + '\n')

print("Saved", len(all_qa_pairs), "pairs")

Saved 91 pairs


In [23]:
random.shuffle(all_qa_pairs)
split_idx = int(len(all_qa_pairs) * 0.85)
train_pairs = all_qa_pairs[:split_idx]
test_pairs = all_qa_pairs[split_idx:]

print(f"Train: {len(train_pairs)}, Test: {len(test_pairs)}")

with open('train.jsonl', 'w') as f:
    for pair in train_pairs:
        f.write(json.dumps(pair) + '\n')

with open('test.jsonl', 'w') as f:
    for pair in test_pairs:
        f.write(json.dumps(pair) + '\n')

Train: 77, Test: 14


In [25]:
!pip install unsloth


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.1/81.1 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.4/24.4 MB 85.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 19.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 27.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 67.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 23.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 64.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 83.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 66.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.9/216.9 kB 25.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.7/119.7 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 199.3/1

In [26]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = None  # auto-detect

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/llama-3-8b-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = True,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.9.11: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth: Will load unsloth/llama-3-8b-bnb-4bit as a legacy tokenizer.


In [27]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,  # rank — size of the "notebook", higher = more capacity but slower
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                       "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 42,
)

Unsloth 2026.9.11 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


In [28]:
model.print_trainable_parameters()

trainable params: 41,943,040 || all params: 8,072,204,288 || trainable%: 0.5196


In [29]:
import os
print(os.path.exists('train.jsonl'), os.path.exists('test.jsonl'))

True True


In [30]:
from datasets import load_dataset

alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

EOS_TOKEN = tokenizer.eos_token

def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs = examples["input"]
    outputs = examples["output"]
    texts = []
    for instruction, input_, output in zip(instructions, inputs, outputs):
        text = alpaca_prompt.format(instruction, input_, output) + EOS_TOKEN
        texts.append(text)
    return {"text": texts}

dataset = load_dataset("json", data_files="train.jsonl", split="train")
dataset = dataset.map(formatting_prompts_func, batched=True)

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/77 [00:00<?, ? examples/s]

In [31]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60,        # enough for a 77-example demo dataset
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        output_dir = "outputs",
    ),
)


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/77 [00:00<?, ? examples/s]

In [32]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 77 | Num Epochs = 6 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!
/usr/local/lib/python3.13/dist-packages/unsloth/import_fixes.py:4091: UserWarning: 'has_cuda' is deprecated, please use 'torch.backends.cuda.is_built()'
  return original(name)
/usr/local/lib/python3.13/dist-packages/unsloth/import_fixes.py:4091: UserWarning: 'has_cudnn' is deprecated, please use 'torch.backends.cudnn.is_available()'
  return original(name)
/usr/local/lib/python3.13/dist-packages/unsloth/import_fixes.py:4091: UserWarning: 'has_mps' is deprecated, please use 'torch.backends.mps.is_built()'
  return original(name)
/usr/local/lib/python3.13/dist-packages/

Step,Training Loss
1,3.077505
2,3.078535
3,2.860061
4,2.715772
5,2.448277
6,2.070872
7,1.628273
8,1.481512
9,1.263788
10,1.023088


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-60/tokenizer_config.json.


In [33]:
import json

FastLanguageModel.for_inference(model)

test_data = []
with open('test.jsonl', 'r') as f:
    for line in f:
        test_data.append(json.loads(line))

print(f"Loaded {len(test_data)} test questions")

Loaded 14 test questions


In [34]:
def generate_answer(instruction, input_text=""):
    prompt = alpaca_prompt.format(instruction, input_text, "")
    inputs = tokenizer([prompt], return_tensors="pt").to("cuda")
    outputs = model.generate(**inputs, max_new_tokens=150, use_cache=True)
    decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]
    # extract just the response part, after "### Response:"
    response = decoded.split("### Response:")[-1].strip()
    return response

In [35]:
finetuned_answers = []
for item in test_data:
    answer = generate_answer(item['instruction'], item.get('input', ''))
    finetuned_answers.append(answer)
    print("Q:", item['instruction'])
    print("Fine-tuned A:", answer)
    print("---")

Both `max_new_tokens` (=150) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=150) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: What should an organization do to ensure compliance with audit trail requirements?
Fine-tuned A: To ensure compliance, organizations should establish policies and procedures for audit trails, provide training on their use, and regularly monitor and review them to detect any unauthorized modifications.
---


Both `max_new_tokens` (=150) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: What are operational system checks?
Fine-tuned A: Operational system checks are procedures that verify the correct sequence of operations in a process, ensuring that all steps are taken in the designated order to maintain compliance and quality.
---


Both `max_new_tokens` (=150) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

Q: What does FDA/GxP require for changes made to electronic records?
Fine-tuned A: FDA/GxP requires that all changes to electronic records are documented with audit trails that accurately reflect the date and time of the action.
---


Both `max_new_tokens` (=150) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: How often should operational system checks be reviewed?
Fine-tuned A: Operational system checks should be reviewed regularly, with the frequency determined by organizational policies and the level of risk associated with the processes.
---


Both `max_new_tokens` (=150) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: What should be included in procedures for electronic records?
Fine-tuned A: Procedures for electronic records should include controls for system access, record generation and handling, record integrity checks, and the ability to generate accurate copies of records as required by regulatory agencies.
---
Q: Are audit trails optional under FDA/GxP regulations?
Fine-tuned A: No, audit trails are not optional under FDA/GxP regulations as they are crucial for compliance and investigations.
---


In [36]:
with open('finetuned_answers.json', 'w') as f:
    json.dump(finetuned_answers, f)
print("Saved fine-tuned answers")

Saved fine-tuned answers


In [37]:
model.disable_adapter_layers()

In [38]:
base_answers = []
for item in test_data:
    answer = generate_answer(item['instruction'], item.get('input', ''))
    base_answers.append(answer)
    print("Q:", item['instruction'])
    print("Base A:", answer)
    print("---")

Both `max_new_tokens` (=150) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=150) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: What should an organization do to ensure compliance with audit trail requirements?
Base A: An organization should ensure that all changes made to data are recorded and that the records of these changes are retained for the appropriate amount of time.
---


Both `max_new_tokens` (=150) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: What are operational system checks?
Base A: Operational system checks are checks that are performed to ensure the correct operation of the system. These checks are designed to detect and correct any errors or issues that may arise during the system's operation. The checks are performed regularly, and they are designed to ensure that the system is functioning correctly and that any issues are addressed as quickly as possible.
---


Both `max_new_tokens` (=150) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

Q: What does FDA/GxP require for changes made to electronic records?
Base A: Electronic records are required to be complete and accurate. Any changes made to electronic records should be reflected in a change control system. Electronic records should be maintained in a manner that ensures that they remain complete and accurate over time. This includes ensuring that any changes made to electronic records are properly documented and that the records are backed up regularly to prevent data loss. Electronic records should also be stored in a manner that ensures that they remain accessible and retrievable over time.
---


Both `max_new_tokens` (=150) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: How often should operational system checks be reviewed?
Base A: The operational system checks should be reviewed every 3 months.
---


Both `max_new_tokens` (=150) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: What should be included in procedures for electronic records?
Base A: What should be included in procedures for electronic records?

### Explanation:
The procedures for electronic records should include the following:

1. The purpose of the record should be clearly stated.
2. The record should be accurate and complete.
3. The record should be kept in a secure location.
4. The record should be updated regularly.
5. The record should be destroyed when it is no longer needed.

### Answer:
What should be included in procedures for electronic records?

### Explanation:
The procedures for electronic records should include the following:

1. The purpose of the record should be clearly stated.
2. The record should be accurate and complete.
3. The record should be kept in a secure location.
4. The record should be updated regularly.
5
---
Q: Are audit trails optional under FDA/GxP regulations?
Base A: Audit trails are not optional under FDA/GxP regulations. The FDA's regulations on electroni

In [39]:
model.enable_adapter_layers()

In [40]:
with open('base_answers.json', 'w') as f:
    json.dump(base_answers, f)
print("Saved base answers")

Saved base answers


In [41]:
def judge_answer(question, answer):
    prompt = f"""You are an expert FDA/GxP regulatory compliance evaluator.

Rate the following answer to a compliance question on a scale of 1-5:
1 = incorrect, irrelevant, or empty
2 = partially relevant but vague or inaccurate
3 = acceptable but generic, missing compliance-specific detail
4 = accurate and reasonably specific
5 = accurate, specific, and uses correct regulatory terminology

Question: {question}
Answer: {answer}

Respond with ONLY a single integer 1-5, nothing else."""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )
    try:
        return int(response.choices[0].message.content.strip())
    except:
        return None

In [42]:
base_scores = []
for item, answer in zip(test_data, base_answers):
    score = judge_answer(item['instruction'], answer)
    base_scores.append(score)
    print(f"{score} — {item['instruction'][:60]}")

print(f"\nBase model average: {sum(s for s in base_scores if s) / len(base_scores):.2f} / 5")

3 — What should an organization do to ensure compliance with aud
3 — What are operational system checks?
3 — What are the consequences of inadequate system validation?
3 — What are the consequences of not implementing authority chec
3 — Why are operational system checks important for compliance?
2 — Who is responsible for managing authority checks?
4 — How long must audit trail documentation be retained accordin
2 — How can I ensure that my procedures are effective?
3 — What actions should be taken to prepare for an inspection of
2 — What is the purpose of enforcing permitted sequencing?
4 — What does FDA/GxP require for changes made to electronic rec
2 — How often should operational system checks be reviewed?
2 — What should be included in procedures for electronic records
5 — Are audit trails optional under FDA/GxP regulations?

Base model average: 2.93 / 5


In [43]:
finetuned_scores = []
for item, answer in zip(test_data, finetuned_answers):
    score = judge_answer(item['instruction'], answer)
    finetuned_scores.append(score)
    print(f"{score} — {item['instruction'][:60]}")

print(f"\nFine-tuned model average: {sum(s for s in finetuned_scores if s) / len(finetuned_scores):.2f} / 5")

4 — What should an organization do to ensure compliance with aud
4 — What are operational system checks?
3 — What are the consequences of inadequate system validation?
4 — What are the consequences of not implementing authority chec
3 — Why are operational system checks important for compliance?
3 — Who is responsible for managing authority checks?
4 — How long must audit trail documentation be retained accordin
3 — How can I ensure that my procedures are effective?
3 — What actions should be taken to prepare for an inspection of
4 — What is the purpose of enforcing permitted sequencing?
4 — What does FDA/GxP require for changes made to electronic rec
3 — How often should operational system checks be reviewed?
4 — What should be included in procedures for electronic records
4 — Are audit trails optional under FDA/GxP regulations?

Fine-tuned model average: 3.57 / 5


In [44]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                       "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 42,
)

Unsloth: Already have LoRA adapters! We shall skip this step.


In [45]:
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 120,       # doubled from 60
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        output_dir = "outputs",
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/77 [00:00<?, ? examples/s]

In [46]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 77 | Num Epochs = 12 | Total steps = 120
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)


Step,Training Loss
1,0.155300
2,0.136282
3,0.143353
4,0.172455
5,0.167748
6,0.182321
7,0.229090
8,0.276970
9,0.255952
10,0.217660


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-120/tokenizer_config.json.


In [47]:
FastLanguageModel.for_inference(model)

finetuned_answers_v2 = []
for item in test_data:
    answer = generate_answer(item['instruction'], item.get('input', ''))
    finetuned_answers_v2.append(answer)
    print("Q:", item['instruction'])
    print("Fine-tuned A:", answer)
    print("---")

Both `max_new_tokens` (=150) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=150) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: What should an organization do to ensure compliance with audit trail requirements?
Fine-tuned A: Organization should implement policies and procedures to ensure that all actions generating audit trails are documented, that records are accurate and complete, and that they can demonstrate compliance with audit trail regulations at any time.
---


Both `max_new_tokens` (=150) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: What are operational system checks?
Fine-tuned A: Operational system checks are mechanisms used to ensure that all operational steps are performed in the correct order, thereby minimizing the risk of deviations that could affect product quality.
---


Both `max_new_tokens` (=150) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

Q: What does FDA/GxP require for changes made to electronic records?
Fine-tuned A: FDA/GxP requires that changes to electronic records are documented with audit trails that accurately reflect the date and time of actions taken.
---


Both `max_new_tokens` (=150) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: How often should operational system checks be reviewed?
Fine-tuned A: Operational system checks should be reviewed regularly, at least annually, or whenever there are changes in processes or regulatory requirements to ensure they remain effective and compliant.
---


Both `max_new_tokens` (=150) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: What should be included in procedures for electronic records?
Fine-tuned A: Procedures for electronic records should include system validation, access restrictions, the ability to generate accurate copies, and the implementation of controls to ensure the integrity of the records.
---
Q: Are audit trails optional under FDA/GxP regulations?
Fine-tuned A: No, audit trails are mandatory under FDA/GxP regulations to ensure compliance and adherence to strict protocols for maintaining electronic records.
---


In [48]:
finetuned_scores_v2 = []
for item, answer in zip(test_data, finetuned_answers_v2):
    score = judge_answer(item['instruction'], answer)
    finetuned_scores_v2.append(score)
    print(f"{score} — {item['instruction'][:60]}")

print(f"\nFine-tuned v2 average: {sum(s for s in finetuned_scores_v2 if s) / len(finetuned_scores_v2):.2f} / 5")

3 — What should an organization do to ensure compliance with aud
4 — What are operational system checks?
3 — What are the consequences of inadequate system validation?
4 — What are the consequences of not implementing authority chec
3 — Why are operational system checks important for compliance?
4 — Who is responsible for managing authority checks?
2 — How long must audit trail documentation be retained accordin
4 — How can I ensure that my procedures are effective?
3 — What actions should be taken to prepare for an inspection of
3 — What is the purpose of enforcing permitted sequencing?
4 — What does FDA/GxP require for changes made to electronic rec
4 — How often should operational system checks be reviewed?
4 — What should be included in procedures for electronic records
5 — Are audit trails optional under FDA/GxP regulations?

Fine-tuned v2 average: 3.57 / 5


In [49]:
import os
for root, dirs, files in os.walk('outputs'):
    for f in files:
        print(os.path.join(root, f))

outputs/README.md
outputs/checkpoint-120/training_args.bin
outputs/checkpoint-120/adapter_config.json
outputs/checkpoint-120/README.md
outputs/checkpoint-120/adapter_model.safetensors
outputs/checkpoint-120/optimizer.pt
outputs/checkpoint-120/scheduler.pt
outputs/checkpoint-120/trainer_state.json
outputs/checkpoint-120/scaler.pt
outputs/checkpoint-120/tokenizer_config.json
outputs/checkpoint-120/tokenizer.json
outputs/checkpoint-120/rng_state.pth
outputs/checkpoint-60/training_args.bin
outputs/checkpoint-60/adapter_config.json
outputs/checkpoint-60/README.md
outputs/checkpoint-60/adapter_model.safetensors
outputs/checkpoint-60/optimizer.pt
outputs/checkpoint-60/scheduler.pt
outputs/checkpoint-60/trainer_state.json
outputs/checkpoint-60/scaler.pt
outputs/checkpoint-60/tokenizer_config.json
outputs/checkpoint-60/tokenizer.json
outputs/checkpoint-60/rng_state.pth


In [50]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                       "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 42,
)

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        output_dir = "outputs",
    ),
)

trainer_stats = trainer.train()

Unsloth: Already have LoRA adapters! We shall skip this step.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 77 | Num Epochs = 6 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)


Step,Training Loss
1,0.060846
2,0.057225
3,0.060546
4,0.068279
5,0.099700
6,0.104021
7,0.094876
8,0.105414
9,0.119951
10,0.106592


In [51]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/llama-3-8b-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                       "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 42,
)

==((====))==  Unsloth 2026.9.11: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth: Will load unsloth/llama-3-8b-bnb-4bit as a legacy tokenizer.


In [52]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                       "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 42,
)

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        output_dir = "outputs",
    ),
)

trainer_stats = trainer.train()

Unsloth: Already have LoRA adapters! We shall skip this step.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 77 | Num Epochs = 6 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)


Step,Training Loss
1,3.077505
2,3.078535
3,2.860147
4,2.716079
5,2.448097
6,2.070560
7,1.628770
8,1.481660
9,1.264307
10,1.023056


In [53]:
FastLanguageModel.for_inference(model)

finetuned_answers_final = []
for item in test_data:
    answer = generate_answer(item['instruction'], item.get('input', ''))
    finetuned_answers_final.append(answer)

finetuned_scores_final = []
for item, answer in zip(test_data, finetuned_answers_final):
    score = judge_answer(item['instruction'], answer)
    finetuned_scores_final.append(score)
    print(f"{score} — {item['instruction'][:60]}")

print(f"\nFinal fine-tuned average: {sum(s for s in finetuned_scores_final if s) / len(finetuned_scores_final):.2f} / 5")

Both `max_new_tokens` (=150) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=150) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=150) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=150) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generati

4 — What should an organization do to ensure compliance with aud
4 — What are operational system checks?
3 — What are the consequences of inadequate system validation?
4 — What are the consequences of not implementing authority chec
3 — Why are operational system checks important for compliance?
3 — Who is responsible for managing authority checks?
4 — How long must audit trail documentation be retained accordin
4 — How can I ensure that my procedures are effective?
3 — What actions should be taken to prepare for an inspection of
4 — What is the purpose of enforcing permitted sequencing?
4 — What does FDA/GxP require for changes made to electronic rec
3 — How often should operational system checks be reviewed?
4 — What should be included in procedures for electronic records
4 — Are audit trails optional under FDA/GxP regulations?

Final fine-tuned average: 3.64 / 5


In [54]:
with open('finetuned_answers.json', 'w') as f:
    json.dump(finetuned_answers_final, f)
print("Saved final fine-tuned answers")

Saved final fine-tuned answers


In [55]:
!pip install huggingface_hub

In [57]:
from huggingface_hub import login
login()

In [58]:
model.push_to_hub("rajeshvctec/llama3-8b-fda-gxp-qlora", token=True)
tokenizer.push_to_hub("rajeshvctec/llama3-8b-fda-gxp-qlora", token=True)

README.md:   0%|          | 0.00/556 [00:00<?, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   0%|          |  559kB /  168MB            

Saved model to https://huggingface.co/rajeshvctec/llama3-8b-fda-gxp-qlora


Unsloth: Restored added_tokens_decoder metadata in /tmp/tmp8calttq2/tokenizer_config.json.


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mp8calttq2/tokenizer.json: 100%|##########| 17.2MB / 17.2MB            